[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/47_lora_data_solution.ipynb)

#  Solution: LoRA Data Preparation

Reference solution for preparing data for LoRA fine-tuning  used extensively in PEFT (Parameter-Efficient Fine-Tuning) pipelines.

```text
1. Format with prompt template: "{instruction}\n\n{completion}"
2. Tokenize full prompt+completion
3. Tokenize prompt-only to find boundary
4. Mask prompt tokens as -100 (only compute loss on completion)
5. Pad/truncate, return batched tensors
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def prepare_lora_data(data, tokenizer, max_length=512, prompt_template=None):
    """Prepare data for LoRA fine-tuning.
    
    Inspired by HuggingFace PEFT, LLaMA-Factory, and Axolotl's LoRA data pipelines.
    Only completion tokens contribute to loss  the prompt is treated as context.
    This matches the standard causal LM training objective for instruction tuning.
    """
    if prompt_template is None:
        prompt_template = "{instruction}\n\n{completion}"
    
    pad_id = getattr(tokenizer, 'pad_token_id', 0)
    all_input_ids = []
    all_labels = []
    
    for sample in data:
        instruction = sample["instruction"]
        completion = sample["completion"]
        
        # Format full text
        full_text = prompt_template.format(instruction=instruction, completion=completion)
        
        # Format prompt only (empty completion) to find boundary
        prompt_text = prompt_template.format(instruction=instruction, completion="")
        
        # Tokenize both
        full = tokenizer(full_text, max_length=max_length, truncation=True)
        full_ids = full["input_ids"]
        
        prompt = tokenizer(prompt_text, max_length=max_length, truncation=True)
        prompt_ids = prompt["input_ids"]
        
        # Determine prompt boundary
        if 0 < len(prompt_ids) < len(full_ids):
            split_point = len(prompt_ids)
        else:
            # Fallback: proportional split
            ratio = len(prompt_text) / max(1, len(full_text))
            split_point = max(1, round(len(full_ids) * ratio))
        
        # Create labels: mask prompt, keep completion
        labels = full_ids.clone().to(dtype=torch.long)
        labels[:split_point] = -100
        
        # Pad or truncate
        if len(full_ids) < max_length:
            pad_len = max_length - len(full_ids)
            full_ids = torch.cat([full_ids, torch.full((pad_len,), pad_id, dtype=full_ids.dtype)])
            labels = torch.cat([labels, torch.full((pad_len,), -100, dtype=labels.dtype)])
        elif len(full_ids) > max_length:
            full_ids = full_ids[:max_length]
            labels = labels[:max_length]
        
        all_input_ids.append(full_ids)
        all_labels.append(labels)
    
    return {
        "input_ids": torch.stack(all_input_ids),
        "labels": torch.stack(all_labels),
    }

In [ ]:
#  Verify

class SimpleTokenizer:
    def __init__(self):
        self.pad_token_id = 0
    def __call__(self, text, max_length=None, truncation=False):
        ids = [ord(c) for c in text] + [2]
        if max_length and len(ids) > max_length:
            ids = ids[:max_length-1] + [2]
        return {"input_ids": torch.tensor(ids)}

data = [{"instruction": "Hi", "completion": "Hello there!"}]
result = prepare_lora_data(data, SimpleTokenizer(), max_length=32)
print("input_ids:", result["input_ids"])
print("labels:   ", result["labels"])
# Prompt region should be -100, completion region should have real token IDs
labels = result["labels"][0]
print(f"First 5 labels: {labels[:5]}")
print(f"Last labels where != -100: {labels[labels != -100]}")

In [ ]:
# Run judge
from torch_judge import check
check("lora_data")